# Étape 2 : Nettoyage et Standardisation des données

Dans cette étape, nous allons préparer les données pour les algorithmes d'apprentissage non supervisé (Clustering : K-Means, DBSCAN).

**Objectifs :**
1. **Nettoyage** : S'assurer qu'il n'y a plus de valeurs manquantes (les codes `97, 98, 99` ont déjà été gérés) et isoler les labels (Résultat COVID) pour ne pas tricher lors du clustering.
2. **Gestion de l'Âge** : Éviter que l'âge (seule variable continue) n'écrase les autres variables dans le calcul des distances. On le transforme en 4 variables binaires (One-Hot Encoding).
3. **Standardisation binaire** : Ramener toutes les comorbidités (actuellement codées 1=Oui, 2=Non) à un format binaire classique (1=Oui, 0=Non).

À la fin, notre dataset sera composé **uniquement de 0 et de 1**, garantissant un poids parfaitement égal pour chaque caractéristique clinique.

In [1]:
import pandas as pd
import numpy as np

# ── 1. Chargement des données Silver ──────────────────────────────────────
SILVER_PATH = '../../data/layer_silver_data_transform/covid_cleaned.csv'
df = pd.read_csv(SILVER_PATH)

print(f"Données chargées : {df.shape[0]:,} patients, {df.shape[1]} colonnes.")

Données chargées : 260,919 patients, 15 colonnes.


### Nettoyage et séparation des labels
Les données provenant de notre couche Silver sont déjà propres (sans `97, 98, 99`). On s'assure de supprimer les éventuelles lignes avec `NaN` par sécurité.
Ensuite, le clustering étant **non supervisé**, nous devons retirer la variable cible (`Resultat_COVID`). Nous la conservons dans une variable à part pour pouvoir analyser nos clusters à l'étape 7.

In [2]:
# ── Nettoyage et Labels ───────────────────────────────────────────────────
# Au cas où, on supprime les lignes avec NaN (stratégie : suppression directe car dataset massif)
df.dropna(inplace=True)

# Sauvegarde des labels pour l'explication finale (Étape 7)
labels_covid = df['Resultat_COVID'].copy()

# Suppression de la cible du dataset d'entraînement
df_cluster = df.drop(columns=['Resultat_COVID']).copy()

print("Aperçu des données d'entraînement (sans labels) :")
df_cluster.head(3)

Aperçu des données d'entraînement (sans labels) :


,Sexe,Type_de_patient,Pneumonie,Age,Diabète,Bronchopneumopathie_chronique_obstructive,Asthme,Immunosuppression,Hypertension,Autre_comorbidité,Maladie_cardiovasculaire,Obésité,Insuffisance_rénale_chronique,Tabagisme
0,2,1,2.0,74,1.0,2.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,2.0
1,1,2,2.0,71,1.0,1.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,1.0
2,2,2,1.0,50,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0


### Gestion de l'âge (Discrétisation et One-Hot Encoding)
Si l'on utilise un `StandardScaler` sur l'âge, on obtiendra des valeurs comme -1.5, 0.2, 2.1, tandis que les comorbidités seront à 0 ou 1. L'âge risquerait encore de dominer légèrement.

La meilleure approche est de **discrétiser** l'âge en tranches (Enfant, Jeune, Adulte, Senior) puis d'appliquer un **One-Hot Encoding**.

In [3]:
# ── Discrétisation de l'Âge ───────────────────────────────────────────────
bins = [-1, 17, 39, 59, 150]
noms_tranches = ['tranche_age_enfant', 'tranche_age_jeune', 'tranche_age_adulte', 'tranche_age_senior']

# Découpage en catégories
df_cluster['Tranche_Age'] = pd.cut(df_cluster['Age'], bins=bins, labels=noms_tranches)

# One-Hot Encoding : convertit les 4 catégories en 4 colonnes de 0 et 1
df_cluster = pd.get_dummies(df_cluster, columns=['Tranche_Age'], dtype=int)

# On supprime la colonne Age originale numérique
df_cluster.drop(columns=['Age'], inplace=True)

print("Colonnes liées à l'âge après encodage :")
print([c for c in df_cluster.columns if 'tranche' in c])

Colonnes liées à l'âge après encodage :
['Tranche_Age_tranche_age_enfant', 'Tranche_Age_tranche_age_jeune', 'Tranche_Age_tranche_age_adulte', 'Tranche_Age_tranche_age_senior']


### Standardisation des comorbidités (Ré-encodage 1/2 en 1/0)
Dans le système SISVER mexicain, les réponses binaires sont encodées avec `1 = Oui` et `2 = Non`. (Sauf Sexe: 1=Femme/2=Homme et Type: 1=Ambu/2=Hospit).
Pour un algorithme mathématique, `0` (Absence) et `1` (Présence) est beaucoup plus sain. On remplace donc tous les `2` par des `0`.

In [4]:
# ── Standardisation Binaire 0/1 ───────────────────────────────────────────
# Liste des colonnes originales (qui ont des valeurs 1 et 2)
cols_originales = [c for c in df_cluster.columns if not c.startswith('Tranche_Age_')]

# Remplacement de la valeur 2 par 0
df_cluster[cols_originales] = df_cluster[cols_originales].replace(2, 0)

print("Vérification : min et max de toutes les colonnes (doivent tous être 0 et 1)")
print(df_cluster.describe().loc[['min', 'max']])

Vérification : min et max de toutes les colonnes (doivent tous être 0 et 1)
     Sexe  Type_de_patient  Pneumonie  Diabète  \
min   0.0              0.0        0.0      0.0   
max   1.0              1.0        1.0      1.0   

     Bronchopneumopathie_chronique_obstructive  Asthme  Immunosuppression  \
min                                        0.0     0.0                0.0   
max                                        1.0     1.0                1.0   

     Hypertension  Autre_comorbidité  Maladie_cardiovasculaire  Obésité  \
min           0.0                0.0                       0.0      0.0   
max           1.0                1.0                       1.0      1.0   

     Insuffisance_rénale_chronique  Tabagisme  Tranche_Age_tranche_age_enfant  \
min                            0.0        0.0                             0.0   
max                            1.0        1.0                             1.0   

     Tranche_Age_tranche_age_jeune  Tranche_Age_tranche_age_adulte  \
m

### Bilan et Export (Couche Gold)
Notre dataset est maintenant **parfaitement standardisé**. Toutes les variables sont sur une échelle binaire stricte [0, 1]. Le calcul de distance du K-Means traitera chaque caractéristique avec la même importance.

On exporte ce dataset dans la couche `Gold` pour l'étape de clustering.

In [5]:
# ── Export pour le Clustering (Layer Gold) ────────────────────────────────
import os
gold_dir = '../../data/layer_gold_data_model'
os.makedirs(gold_dir, exist_ok=True)

# On ajoute temporairement les labels pour les stocker avec les données transformées,
# ou on sauvegarde deux fichiers distincts. Le plus simple est de garder les labels dans le CSV
# final et de faire `drop` juste avant de lancer le `fit` du KMeans.

df_gold = df_cluster.copy()
df_gold['Label_Resultat_COVID'] = labels_covid

output_path = f'{gold_dir}/covid_clustering_ready.csv'
df_gold.to_csv(output_path, index=False)

print(f"Dataset Gold exporté avec succès : {output_path}")

Dataset Gold exporté avec succès : ../../data/layer_gold_data_model/covid_clustering_ready.csv
